In [1]:
!pip -q install easyocr faster-whisper transformers sentencepiece accelerate keybert sentence-transformers gradio pillow opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 9.4 MB/s eta 0:00:00


In [3]:
# Cell 2 — Imports and model loading (FIXED VERSION)

import re
import json
from pathlib import Path

import numpy as np
from PIL import Image

import easyocr
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer, util

# OCR
ocr_reader = easyocr.Reader(["en"], gpu=False)

# Speech-to-text
stt_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

# FLAN-T5 Question Generator (FIXED)
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def run_flan(prompt, max_new_tokens=160):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Keywords
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

# Embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Cell 3 — Helper functions (FIXED VERSION)

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_text_from_image(image_path: str) -> str:
    results = ocr_reader.readtext(image_path, detail=0)
    return clean_text(" ".join(results))

def transcribe_audio(audio_path: str) -> str:
    segments, info = stt_model.transcribe(
        audio_path,
        beam_size=5
    )

    parts = [seg.text for seg in segments]
    return clean_text(" ".join(parts))

def extract_keywords(text: str, top_n: int = 8):
    if len(text.split()) < 5:
        return []

    kws = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n
    )

    return [k for k, _ in kws]

def similarity(a: str, b: str) -> float:
    if not a.strip() or not b.strip():
        return 0.0

    va = embedder.encode(a, convert_to_tensor=True)
    vb = embedder.encode(b, convert_to_tensor=True)

    return float(util.cos_sim(va, vb)[0][0])

# Question generation
def generate_questions(context: str, n: int = 5):

    prompt = f"""
Generate exactly {n} concise interview questions from this project context.

Focus on:
- architecture
- implementation
- model choice
- evaluation
- originality
- limitations

Return only questions.
One question per line.

Context:
{context[:3000]}
"""

    raw = run_flan(prompt)

    lines = [
        re.sub(r"^\d+[\).\s-]*", "", x).strip()
        for x in raw.split("\n")
    ]

    questions = [
        x for x in lines
        if len(x) > 10
    ]

    fallback = [
        "Can you explain the architecture of your project?",
        "What problem does your project solve?",
        "Why did you choose these models or libraries?",
        "What implementation challenges did you face?",
        "How did you evaluate your system?",
        "What are the limitations of your solution?",
        "How can the project be improved further?",
        "Which part of the project is most original?"
    ]

    for q in fallback:
        if len(questions) >= n:
            break

        if q not in questions:
            questions.append(q)

    return questions[:n]

# Adaptive follow-up
def generate_followup(question: str, answer: str, context: str):

    prompt = f"""
Question:
{question}

Student answer:
{answer}

Project context:
{context[:1000]}

Generate 1 short follow-up interview question
to probe deeper into technical understanding.

Return only the question.
"""

    raw = run_flan(prompt)

    return raw.strip()

# Scoring
def score_answer(answer: str, context: str, question: str):

    answer_l = answer.lower()

    kws = extract_keywords(context, top_n=8)

    keyword_coverage = (
        sum(1 for kw in kws if kw.lower() in answer_l)
        / max(len(kws), 1)
    )

    sim_context = similarity(answer, context)
    sim_question = similarity(answer, question)

    technical_depth = round(
        min(
            100,
            35 * sim_context +
            35 * keyword_coverage * 100 +
            30 * sim_question
        ),
        2
    )

    clarity = round(
        min(
            100,
            20 + min(len(answer.split()) / 1.2, 80)
        ),
        2
    )

    originality = round(
        max(0, 100 - 80 * sim_context),
        2
    )

    implementation = round(
        min(
            100,
            40 * keyword_coverage * 100 +
            30 * sim_question +
            30 * min(len(answer.split()) / 40, 1) * 100
        ),
        2
    )

    overall = round(
        0.35 * technical_depth +
        0.20 * clarity +
        0.15 * originality +
        0.30 * implementation,
        2
    )

    if overall >= 80:
        feedback = (
            "Very strong answer with good technical depth."
        )

    elif overall >= 60:
        feedback = (
            "Good answer but needs more implementation details."
        )

    else:
        feedback = (
            "Weak answer. Add clearer technical explanation."
        )

    return {
        "technical_depth": technical_depth,
        "clarity": clarity,
        "originality": originality,
        "implementation_understanding": implementation,
        "overall": overall,
        "feedback": feedback
    }

In [6]:
# Cell 4 — Input files

# Replace these with your actual files in Colab
image_path = "/content/Image.png"   # screenshot / slide image
audio_path = "/content/u_t60o3simia-hanuman-song-maker-song-abhi-404570.mp3"    # student speech audio

assert Path(image_path).exists(), f"Image not found: {image_path}"
assert Path(audio_path).exists(), f"Audio not found: {audio_path}"

print("Files found successfully.")

Files found successfully.


In [7]:
# Cell 5 — Extract OCR and transcript

ocr_text = extract_text_from_image(image_path)
stt_text = transcribe_audio(audio_path)

project_context = clean_text(ocr_text + " " + stt_text)

print("OCR TEXT:\n", ocr_text[:1200])
print("\nSTT TEXT:\n", stt_text[:1200])
print("\nMERGED CONTEXT LENGTH:", len(project_context))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR TEXT:
 Machine_learning_eng_Rahul_ res X File C/Users/RahulOneDrive%20-%2OIndian%2Olnstitute%2Oof%2OTechnology?2OIndian%2OSchool%2Oof%2OMines%2ODhanbad/Desktop/Machine_learning_eng_Rahul_ 0 Chat Folder Google Dr; Suggested Sites Gmail Maps Sarkaru Vaari Paata . Vegamovies NL - 3_ B-TECH MINING S_ Previous year Mid_. in search yahoo.com Other favorites Draw 0 A^ a6 Ask Copilot of 2 CD 16 0 R | 1 & Edit with Acrobat RAHUL KUMAR SHARMA NOIDA; UTTAR PRADESH, INDIA J+919508476508 kumarsharmarahul0l@gmail.com in} linkedin com/in/rahul-kumar-sharma-aa0b5 7233 github.com/Rahul29999 Education Indian Institute of Technology (ISM) Dhanbad Oct 2021 May 2025 Bachelor of Technology in Mining Engineering Dhanbad Jharkhand Relevant Coursework: Artificial Intelligence, Machine Learning (Supervised Unsupervised) , Decp Learning, Natural Language Processing; Probability & Statistics Data Analysis, Data Visualization, Python Programming: Experience Outlier AI Nov 2024 Nov 2025 Machine Learning Enginee

In [8]:
# Cell 6 — Generate interview questions

questions = generate_questions(project_context, n=5)

print("Generated Questions:\n")
for i, q in enumerate(questions, 1):
    print(f"{i}. {q}")

Generated Questions:

1. What is the name of the institute that provides training for machine learning?
2. Can you explain the architecture of your project?
3. What problem does your project solve?
4. Why did you choose these models or libraries?
5. What implementation challenges did you face?


In [9]:
# Cell 7 — Interactive interview loop with adaptive follow-up

results = []
question_history = []

for i, q in enumerate(questions, 1):

    print(f"\nQ{i}: {q}")

    answer = input("Your answer: ").strip()

    result = score_answer(
        answer,
        project_context,
        q
    )

    # Adaptive follow-up condition
    if result["overall"] < 70 or len(answer.split()) < 25:

        followup = generate_followup(
            q,
            answer,
            project_context
        )

        print("\nFollow-up Question:")
        print(followup)

        followup_answer = input(
            "Your follow-up answer: "
        ).strip()

        result["followup_question"] = followup
        result["followup_answer"] = followup_answer

    else:

        result["followup_question"] = ""
        result["followup_answer"] = ""

    result["question"] = q
    result["answer"] = answer

    results.append(result)
    question_history.append(q)

print("\nInterview completed successfully.")


Q1: What is the name of the institute that provides training for machine learning?
Your answer: Indian institiute of technology (ism) dhanbad

Follow-up Question:
What is the full name of the Institute of Technology in Dhanbad?
Your follow-up answer: Indian institute of technology (indian school of mines ) dhanbad 

Q2: Can you explain the architecture of your project?
Your answer: The architecture of my project consists of four main modules. First is the OCR module using EasyOCR, which extracts text from presentation slides and screen content. Second is the speech-to-text module using Faster-Whisper that converts student speech into text. Third is the context analysis and question generation module using FLAN-T5, which generates interview questions from the extracted project context. Fourth is the evaluation module that scores answers based on technical depth, clarity, originality, and implementation understanding using semantic similarity and keyword analysis.

Follow-up Question:
W

In [10]:
# Cell 8 — Print score summary

import pandas as pd

df = pd.DataFrame(results)[[
    "question",
    "overall",
    "technical_depth",
    "clarity",
    "originality",
    "implementation_understanding",
    "feedback"
]]

display(df)

print("\nAverage Score:", round(df["overall"].mean(), 2))

,question,overall,technical_depth,clarity,originality,implementation_understanding,feedback
0,What is the name of the institute that provide...,51.87,23.33,25.00,58.00,100,Weak answer. Add clearer technical explanation.
1,Can you explain the architecture of your project?,67.44,21.77,88.33,81.06,100,Good answer but needs more implementation deta...
2,What problem does your project solve?,58.73,18.73,49.17,82.29,100,Weak answer. Add clearer technical explanation.
3,Why did you choose these models or libraries?,61.85,12.67,70.00,89.41,100,Good answer but needs more implementation deta...
4,What implementation challenges did you face?,62.51,17.97,67.50,84.80,100,Good answer but needs more implementation deta...



Average Score: 60.48


In [12]:
# Cell 9 — Generate PDF report

!pip -q install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def generate_report(results, student="Student", project="AI-Driven Automated Interviewer"):
    doc = SimpleDocTemplate("interview_report.pdf", pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph(project, styles["Title"]))
    story.append(Paragraph(f"Student: {student}", styles["Normal"]))
    story.append(Spacer(1, 12))

    avg_score = sum(r["overall"] for r in results) / max(len(results), 1)
    story.append(Paragraph(f"Overall Score: {avg_score:.2f}/100", styles["Heading2"]))
    story.append(Spacer(1, 10))

    summary_data = [["Metric", "Average"]]
    for key in ["technical_depth", "clarity", "originality", "implementation_understanding"]:
        avg_val = sum(r[key] for r in results) / max(len(results), 1)
        summary_data.append([key.replace("_", " ").title(), f"{avg_val:.2f}"])

    table = Table(summary_data, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("PADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(table)
    story.append(Spacer(1, 14))

    for idx, r in enumerate(results, 1):
        story.append(Paragraph(f"Q{idx}: {r['question']}", styles["Heading3"]))
        story.append(Paragraph(f"Answer: {r['answer']}", styles["Normal"]))

        if r.get("followup_question"):
            story.append(Paragraph(f"Follow-up: {r['followup_question']}", styles["Normal"]))
            story.append(Paragraph(f"Follow-up Answer: {r['followup_answer']}", styles["Normal"]))

        story.append(Paragraph(f"Score: {r['overall']}/100", styles["Normal"]))
        story.append(Paragraph(f"Feedback: {r['feedback']}", styles["Normal"]))
        story.append(Spacer(1, 10))

    doc.build(story)
    print("PDF saved: interview_report.pdf")

generate_report(results, student="Rahul Kumar Sharma")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.9 MB/s eta 0:00:00
PDF saved: interview_report.pdf
